In [91]:
import enum
import math
import json
import random
from typing import List, Tuple

EPSILON = 1e-9

In [ ]:
class Direction(enum.Enum):
    UP = 0
    RIGHT = 1
    DOWN = 2
    LEFT = 3

    def rotate_clockwise(self):
        return Direction((self.value + 1) % 4)

    def rotate_counterclockwise(self):
        return Direction((self.value - 1) % 4)

    def symmetric_NW(self):
        if self == Direction.UP:
            return Direction.LEFT
        elif self == Direction.RIGHT:
            return Direction.DOWN
        elif self == Direction.DOWN:
            return Direction.RIGHT
        elif self == Direction.LEFT:
            return Direction.UP

    def symmetric_NE(self):
        if self == Direction.UP:
            return Direction.RIGHT
        elif self == Direction.RIGHT:
            return Direction.UP
        elif self == Direction.DOWN:
            return Direction.LEFT
        elif self == Direction.LEFT:
            return Direction.DOWN

    def to_coordinates(self) -> Tuple[int, int]:
        if self == Direction.UP:
            return (0, 1)
        elif self == Direction.RIGHT:
            return (1, 0)
        elif self == Direction.DOWN:
            return (0, -1)
        elif self == Direction.LEFT:
            return (-1, 0)
        raise ValueError("Invalid Direction")

In [ ]:
def make_wire(order: int):
    if order == 0:
        return []
    else:
        prev_wire = make_wire(order - 1)
        new_wire = []
        new_wire += [dir.symmetric_NW() for dir in prev_wire]
        new_wire.append(Direction.LEFT)
        new_wire += prev_wire[::]
        new_wire.append(Direction.UP)
        new_wire += prev_wire[::]
        new_wire.append(Direction.RIGHT)
        new_wire += [dir.symmetric_NE() for dir in prev_wire]
        return new_wire

In [ ]:
def make_origin_wire(order: int) -> List[Direction]:
    wire = make_wire(order)
    return wire[(4**order - 1) // 3 :]

In [ ]:
def get_order(input_length: int) -> int:
    return math.ceil(math.log2((3 * input_length - 1) / 2) / 2 - EPSILON)

In [ ]:
def make_origin_wire(input_length: int) -> List[Direction]:
    order = get_order(input_length)
    wire = make_wire(order)
    i_0 = (4**order - 1) // 3
    return wire[i_0 : i_0 + input_length - 1]

In [88]:
def make_initial_configuration(
    input_word: str,
):
    wire = make_origin_wire(len(input_word))
    i = 0
    x, y = 0, 0
    width = 0
    height = 0
    signals = {f"{x},{y}": [f"input_{input_word[i]}", f"wire_out_{wire[0].name}"]}

    for i in range(1, len(input_word) - 1):
        dx, dy = wire[i - 1].to_coordinates()
        x += dx
        y += dy
        width = max(width, x)
        height = max(height, y)
        signals[f"{x},{y}"] = [
            f"input_{input_word[i]}",
            f"wire_in_{wire[i-1].name}",
            f"wire_out_{wire[i].name}",
        ]

    # last cell
    dx, dy = wire[-1].to_coordinates()
    x += dx
    y += dy
    width = max(width, x)
    height = max(height, y)
    signals[f"{x},{y}"] = [f"input_{input_word[-1]}", f"wire_in_{wire[-1].name}"]
    return {
        "version": "1",
        "size": [width + 1, height + 1],
        "signals": signals,
    }

In [93]:
n = 600
word = "".join([random.choice("AB") for _ in range(n)])
with open("hilbert.json", "w") as f:
    f.write(json.dumps(make_initial_configuration(word), indent=2))